In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
%matplotlib inline
import seaborn as sns

In [2]:
df = pd.read_csv("16-diabetes.csv")

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [13]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [14]:
columns_to_check = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in columns_to_check:
    zero_count = (df[col] == 0).sum()
    zero_percent = 100 * zero_count / len(df)
    print(f"{col}: {zero_count} - %{zero_percent:.2f}")


Glucose: 5 - %0.65
BloodPressure: 35 - %4.56
SkinThickness: 227 - %29.56
Insulin: 374 - %48.70
BMI: 11 - %1.43


"Outcome" and "Insulin" dropping 

In [17]:
X = df.drop(["Outcome", "Insulin"], axis=1)
y = df["Outcome"]

In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 15)

In [20]:
X_train.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age
304,3,150,76,0,21.0,0.207,37
297,0,126,84,29,30.7,0.520,24
522,6,114,0,0,0.0,0.189,26
618,9,112,82,24,28.2,1.282,50
501,3,84,72,32,37.2,0.267,28


In [21]:
columns_to_fill = ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI']

medians = {}
for col in columns_to_fill:
    median_value = X_train[X_train[col] != 0][col].median()
    medians[col] = median_value
    X_train[col] = X_train[col].replace(0, median_value)

for col in columns_to_fill:
    X_test[col] = X_test[col].replace(0, medians[col])

In [22]:
X_test.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age
count,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000,154.000000
mean,3.597403,122.038961,71.487013,29.376623,32.483117,0.479565,33.064935
std,3.304818,32.320876,11.813495,10.513035,6.946159,0.343303,12.118519
min,0.000000,61.000000,30.000000,7.000000,18.400000,0.078000,21.000000
25%,1.000000,95.250000,64.000000,23.250000,26.925000,0.254000,24.000000
50%,3.000000,117.000000,72.000000,29.000000,32.300000,0.376500,28.000000
75%,5.750000,142.750000,80.000000,33.750000,36.950000,0.603750,41.000000
max,13.000000,197.000000,106.000000,99.000000,55.000000,2.329000,69.000000


In [23]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [25]:
def calculate_model_metrics(true, predict):
    class_report = classification_report(true, predict)
    acc_score = accuracy_score(true, predict) 
    con_matrix = confusion_matrix(true, predict) 
    return class_report, acc_score, con_matrix

In [26]:
class_models = {
    "Logistic Regression": LogisticRegression(),
    "KNeighbors Classifier": KNeighborsClassifier(),
    "Decision Tree Classifier": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(),
    "AdaBoost Classifier": AdaBoostClassifier()
}

In [33]:
for i in range(len(list(class_models))):
    model = list(class_models.values())[i]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    model_class_report, model_acc_score, model_con_matrix = calculate_model_metrics(y_test, y_pred)
    
    print(list(class_models.values())[i])
    print("-----------------------------")
    print("Classification Report: \n", model_class_report)
    print("Accuracy Score:", model_acc_score)
    print("Confusion Matrix: \n", model_con_matrix)
    print("----------------------------")

LogisticRegression()
-----------------------------
Classification Report: 
               precision    recall  f1-score   support

           0       0.82      0.83      0.83       108
           1       0.59      0.57      0.58        46

    accuracy                           0.75       154
   macro avg       0.70      0.70      0.70       154
weighted avg       0.75      0.75      0.75       154

Accuracy Score: 0.7532467532467533
Confusion Matrix: 
 [[90 18]
 [20 26]]
----------------------------
KNeighborsClassifier()
-----------------------------
Classification Report: 
               precision    recall  f1-score   support

           0       0.82      0.76      0.79       108
           1       0.52      0.61      0.56        46

    accuracy                           0.71       154
   macro avg       0.67      0.68      0.67       154
weighted avg       0.73      0.71      0.72       154

Accuracy Score: 0.7142857142857143
Confusion Matrix: 
 [[82 26]
 [18 28]]
---------------